In [ ]:
# This cell finds the most recent of the "PreCal SSH unsmoothed" files - these are the most near-real-time with the data we need.
# All files are "half-orbits" from (near) one pole to the other. There's obviously a low of lower-latitude wasted stuff in each file. C'est la vie.

import requests
from datetime import datetime, timezone

short_name = "SWOT_L2_LR_PreCalSSH_UNSMOOTHED_D"

r = requests.get(
    "https://cmr.earthdata.nasa.gov/search/granules.json",
    params={
        "short_name": short_name,
        "page_size": 1,
        "sort_key[]": "-start_date",
    }
)
r.raise_for_status()

g = r.json()["feed"]["entry"][0]

end = datetime.fromisoformat(g["time_end"].replace("Z", "+00:00"))
age = (datetime.now(timezone.utc) - end).total_seconds() / 3600

print("Granule:", g["producer_granule_id"])
print("Start:  ", g["time_start"])
print("End:    ", g["time_end"])
print(f"Age:     {age:.1f} hours")

In [ ]:
import requests
from pathlib import Path

granule_id = (
    "SWOT_L2_LR_PreCalSSH_Unsmoothed_056_142_"
    "20260915T194710_20260915T203836_PID0_01"
)

# Find the granule
r = requests.get(
    "https://cmr.earthdata.nasa.gov/search/granules.json",
    params={
        "short_name": "SWOT_L2_LR_PreCalSSH_UNSMOOTHED_D",
        "producer_granule_id": granule_id,
    }
)
r.raise_for_status()

entries = r.json()["feed"]["entry"]
print(f"Found {len(entries)} granule(s)")

g = entries[0]

# Find the NetCDF download URL
download_url = next(
    link["href"]
    for link in g["links"]
    if link.get("href", "").endswith(".nc")
)

print(download_url)

In [ ]:
filename = Path(download_url.split("/")[-1])

with requests.get(download_url, stream=True) as r:
    r.raise_for_status()

    with open(filename, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)

print("Downloaded:", filename)

In [ ]:
# That was ~750 MB and took a good few minutes to pull down.
import xarray as xr
ds_unsm = xr.open_dataset("/g/data/jk72/af1544/SWOT_nrt_test/SWOT_L2_LR_PreCalSSH_Unsmoothed_056_142_20260915T194710_20260915T203836_PID0_01.nc")

In [ ]:
ds_unsm

In [ ]:
# Weird - the above looks kinda "empty" for 750 MB!
# The trick is there's a 'left' and a 'right' group in those .nc files, which is a bit hidden.

In [ ]:
from netCDF4 import Dataset

nc = Dataset("/g/data/jk72/af1544/SWOT_nrt_test/SWOT_L2_LR_PreCalSSH_Unsmoothed_056_142_20260915T194710_20260915T203836_PID0_01.nc")
print(nc.groups.keys())

In [ ]:
ds_left = xr.open_dataset("/g/data/jk72/af1544/SWOT_nrt_test/SWOT_L2_LR_PreCalSSH_Unsmoothed_056_142_20260915T194710_20260915T203836_PID0_01.nc", group="left")
ds_right = xr.open_dataset("/g/data/jk72/af1544/SWOT_nrt_test/SWOT_L2_LR_PreCalSSH_Unsmoothed_056_142_20260915T194710_20260915T203836_PID0_01.nc", group="right")

print(ds_left)
print(ds_right)

In [ ]:
ds_left

In [ ]:
ds_left.ssha_karin_2.isel(num_lines=slice(78000,80000)).plot(vmin=-1,vmax=1)

In [ ]:
# Hmmm a weird cross-track gradient in ssha. 
x = ds_left.ssha_karin_2.isel(
    num_lines=slice(78000, 79500)
)

x.mean(dim="num_lines").plot()

In [ ]:
import numpy as np
import xarray as xr

x = ds_left.ssha_karin_2.isel(
    num_lines=slice(78200, 79300)
)

# Mean cross-track profile
profile = x.mean("num_lines")

# Fit quadratic, ignoring NaNs, subtract the quadratic fit then plot the residual.
pix = profile.num_pixels.values
z = profile.values
good = np.isfinite(z)

coef = np.polyfit(pix[good], z[good], deg=2)
bias = np.polyval(coef, pix)

bias = xr.DataArray(
    bias,
    coords={"num_pixels": profile.num_pixels},
    dims="num_pixels"
)

x_corr = x - bias

x_corr.plot(vmin=-0.3, vmax=0.3)

In [ ]:
# "ssha_karin" is needed, rather than "ssh_karin" - the "a" is anomaly and means the geoid is subtracted already.
# The "_2" is needed which corresponds to the model-based corrections, rather than the obs-based ones. This _2 behaves much better in sea ice.
